# WSJ 2027 - Gruppindelning Rundresa

Assign confirmed rundresa deltagare into groups of exactly 36.

## Rules
1. **Exactly 36 per group** (remainder group allowed)
2. **Geographic proximity** — participants should live close to each other (Hilbert curve sort)
3. **Friend wish** — at least ONE of friend_1/friend_2 in same group (soft goal)
4. **Max 8 from same kår** per group (hard constraint)
5. **Diversity** — age (14-17) and sex should be as evenly spread as possible

In [1]:
import sys
sys.path.insert(0, '/config/notebooks/wsj27')
import wsj27_utils as u

# Fetch and parse all participants
raw_data = u.fetch_participants()
df, skipped = u.build_participant_dataframe(raw_data)

Fetched 2478 participants
Confirmed: 2413, Unconfirmed: 3, Cancelled: 62
Total confirmed participants: 2413
Skipped: 3 unconfirmed, 0 wrong age/no DOB

By category:
category
deltagare    1940
ist           451
cmt            22

By travel type:
travel
rundresa      1515
direktresa     572
egen_resa      304
other           22

Friend wishes:
  With friend 1 member no: 1358
  With friend 2 member no: 805
  With friend 1 name (text): 118
  With friend 2 name (text): 80


In [2]:
GROUP_SIZE = 36

# Filter to rundresa deltagare only
df_rundresa = df[df['travel'] == 'rundresa'].copy().reset_index(drop=True)
print(f"=== Rundresa deltagare ===")
print(f"Participants: {len(df_rundresa)}")

# Assign coordinates and Hilbert sort
u.assign_coordinates(df_rundresa)
df_sorted = u.add_hilbert_index(df_rundresa)

n_full = len(df_sorted) // GROUP_SIZE
remainder = len(df_sorted) % GROUP_SIZE
total_groups = n_full + (1 if remainder > 0 else 0)
print(f"\nGroups: {n_full} x {GROUP_SIZE} + 1 x {remainder} = {total_groups} total")
print(f"\nBy region:")
print(df_sorted['region'].value_counts().to_string())
print(f"\nBy age:")
print(df_sorted['age'].value_counts().sort_index().to_string())
print(f"\nBy sex:")
print(df_sorted['sex'].map(u.SEX_MAP).value_counts().to_string())

=== Rundresa deltagare ===
Participants: 1515
With coordinates: 1512
Without coordinates (Sweden centroid): 3

Groups: 42 x 36 + 1 x 3 = 43 total

By region:
region
Region Stockholm    533
Region Södra        318
Region Västra       252
Region Norr/Mitt    225
Region Östra        102
                      2

By age:
age
14    308
15    415
16    360
17    285
18     48
19     34
20      9
21     20
22     13
23      8
24      6
25      9

By sex:
sex
Kvinna    763
Man       738
Annat      11
Okänt       3


In [3]:
# Resolve text-only friend wishes via fuzzy name matching
u.resolve_friend_wishes(df_sorted, df)
friend_wishes = u.build_friend_graph(df_sorted)

=== Text Friend Name Matching ===
Text-only wishes: 111
Matched & applied: 78
Generic wishes (not a person): 4
Unresolved (friend not in project): 29

Matched:
  Adelia Vallin -> Leo Norstedt (S:ta Maria Scoutkår) [rundresa] via fuzzy(0.96)
  Albin Åkesson -> Johannes Hellsten (Borlänge Scoutkår) [direktresa] via fuzzy(0.76)
  Albin Åkesson -> Malte Ekström (Lomma scoutkår) [rundresa] via fuzzy(0.84)
  Alex Liljerot Priftis -> Vera Tollwé (Årsta Scoutkår) [rundresa] via fuzzy(0.86)
  Alice Jurensons -> Hedvig Seth (Torslanda Sjöscoutkår) [rundresa] via exact
  Alma Stössel Weinryb -> Lotten Hellman (Södermalms Scoutkår) [rundresa] via exact
  Alma Stössel Weinryb -> Alfons Linder (Hjärnarps Scoutkår) [egen_resa] via fuzzy(0.77)
  Anton Råström -> Arvid Tegel (Trosa Scoutkår) [rundresa] via exact
  Artemis Mörk -> Arthur Gullberg Isaksson (Järlinden Scoutkår) [rundresa] via exact
  Charlie Lindberg -> Axel Lindroth (Saltsjö-Boo Scoutkår) [rundresa] via fuzzy(0.96)
  Dag Jerneck -> Walte

In [4]:
# Run the full group assignment algorithm
df_sorted = u.assign_groups(df_sorted, GROUP_SIZE, friend_wishes)
total_groups = df_sorted['group'].nunique()

Participants: 1515
Groups: 42 x 36 + 1 x 3 = 43 total

=== Phase 1: Geographic sort + cut ===
  Friend satisfaction: 759/945
  Kar violations: 270
  Avg geo spread: 0.6569

=== Phase 2: Fix friend wishes ===
  Swaps: 121
  Friend satisfaction: 933/945
  Kar violations: 240
  Avg geo spread: 1.1240

=== Phase 3: Fix kar violations (geo-aware) ===
  Swaps: 231
  Kar violations: 0
  Friend satisfaction: 690/945
  Avg geo spread: 1.1772

=== Phase 2b: Re-fix friends after kar fix ===
  Swaps: 150
  Friend satisfaction: 927/945
  Kar violations: 0
  Avg geo spread: 1.4113

=== Phase 4: Diversity optimization (geo-penalized) ===
  Swaps: 308
  Diversity score: 140.34 -> 142.16
  Avg geo spread: 1.4113 -> 1.5694

=== FINAL RESULTS ===
Groups: 42 x 36 + 1 x 3
Friend satisfaction: 927/945 (98%)
Kar violations: 0
Total swaps: 810
Diversity: 142.16
Avg geo spread: 1.5694


In [5]:
# Per-group quality metrics
group_of = df_sorted['group'].values
u.print_group_metrics(df_sorted, group_of, total_groups)

Grupp Storlek  Van% MaxKar     M/K/A  14  15  16  17  AvstandKm Karer
--------------------------------------------------------------------------------
    1      36 20/21      8   12/24/0  12   9   7   5        23    16
        Orter: Vellinge (8), Karlsro (6), Torns (4), Hyby (3), Trelleborgs (2), Svedala (2), Husie-Hohögs (2), Dalby (1), Finn (1), Drotten (1), Månstorps (1), Oxie (1), Bunkeflo (1), Klippans (1), Jonstorps (1), Bromölla (1)
    2      36 25/26      8   17/19/0   7  12   7   7        34    14
        Orter: Dalby (8), Karlsro (8), Veberöd (3), Sjöbo KM (3), Södra Sandby (3), Finn (3), Skurups (1), Eslövs (1), Månstorps (1), Påarps (1), Jonstorps (1), Mölndals (1), S:t Örjans Scoutkår Borås (1), Equmenia Scout (1)
    3      36 28/28      7   20/16/0  11   7   5  10        12    11
        Orter: Dalby (7), Karlsro (6), Finn (5), Eslövs (4), Staffanstorps Scoutkår Torparna (4), Tornugglan (2), Drotten (2), Bunkeflo (2), Ottarps (2), Sjöbo KM (1), Scouterna Lödde (1)
   

In [6]:
# Export CSV + JSON
OUTPUT_DIR = '/config/notebooks/wsj27/output'
csv_path, json_path = u.export_results(df_sorted, group_of, total_groups, OUTPUT_DIR, prefix='wsj27_rundresa')

Saved 1515 participants to /config/notebooks/wsj27/output/wsj27_rundresa_grupper.csv
Saved group summary to /config/notebooks/wsj27/output/wsj27_rundresa_grupper.json

CSV preview (first 10 rows):
 group              name member_no  age    sex                    kar                        district           region friend_1 friend_2       lat       lng
     1    Nils Hyddesten   3344654   14    Man      Bromölla Scoutkår         Snapphane Scoutdistrikt     Region Södra                   56.116667 14.550000
     1    Tuva Magnusson   3311571   18 Kvinna      Bunkeflo Scoutkår      Södra Skånes Scoutdistrikt     Region Södra                   55.557482 12.963234
     1  Matilda Knaggård   3296993   20 Kvinna         Dalby Scoutkår      Södra Skånes Scoutdistrikt     Region Södra                   55.664664 13.346159
     1       Alva Landby   3345356   15 Kvinna  Husie-Hohögs Scoutkår      Södra Skånes Scoutdistrikt     Region Södra  3360306          55.583300 13.100000
     1     Isabell

In [7]:
# Generate interactive map
map_path = f'{OUTPUT_DIR}/wsj_rundresa_karta.html'
u.generate_group_map_html(df_sorted, total_groups, map_path, title='WSJ 2027 Rundresa',
                          friend_wishes=friend_wishes, show_group_arcs=True)
print(f"\nAll outputs:")
print(f"  CSV:  {csv_path}")
print(f"  JSON: {json_path}")
print(f"  Map:  {map_path}")

Friend arcs: 965 (780 satisfied, 185 unsatisfied)
Group arcs: 26463 connections across 43 groups
Saved group map: /config/notebooks/wsj27/output/wsj_rundresa_karta.html

All outputs:
  CSV:  /config/notebooks/wsj27/output/wsj27_rundresa_grupper.csv
  JSON: /config/notebooks/wsj27/output/wsj27_rundresa_grupper.json
  Map:  /config/notebooks/wsj27/output/wsj_rundresa_karta.html
